# LeetCode 3911 — K-th Smallest Remaining Even Integer in Subarray Queries bằng Python

- Link: https://leetcode.com/problems/k-th-smallest-remaining-even-integer-in-subarray-queries/
- Độ khó: **Hard**
- Chủ đề: Tìm kiếm nhị phân (`bisect` với `lo/hi`), bài toán "số thứ k bị thiếu", tiền xử lý mảng đơn điệu, truy vấn đoạn

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kthRemainingInteger(self, nums: list[int], queries: list[list[int]]) -> list[int]:
```

## 1. Mục tiêu học

1. Đưa bài về dạng quen thuộc **"số nguyên dương thứ k không có trong một mảng tăng ngặt"** (giống LC 1539 *Kth Missing Positive Number*) bằng phép chia đôi các số chẵn.
2. Hiểu đại lượng `S[i] - i` (số phần tử bị thiếu trước `S[i]`) và vì sao nó **không giảm**, nên tìm nhị phân được.
3. Biết cách **tiền xử lý một lần** mảng `G[j] = H[j] - j` toàn cục rồi trả lời mỗi truy vấn bằng `bisect` trên **đoạn con** `[a, b)` — mẹo dịch chỉ số `k - a`.
4. So sánh với cách **tìm nhị phân trên đáp án** (đếm số chẵn bị xóa `≤ x`).
5. Viết bộ test đối chiếu với mô phỏng trực tiếp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `nums` **tăng ngặt** và danh sách truy vấn `[l, r, k]`. Với mỗi truy vấn:

1. Xét đoạn con `nums[l..r]`.
2. Từ dãy vô hạn các số chẵn dương `2, 4, 6, 8, …`, **xóa** mọi số xuất hiện trong đoạn con đó.
3. Tìm số **nhỏ thứ k** còn lại.

Trả về mảng đáp án cho các truy vấn.

**Ví dụ 1:** `nums = [1,4,7]`, `queries = [[0,2,1],[1,1,2],[0,0,3]]` → `[2,6,6]`.

**Ví dụ 2:** `nums = [2,5,8]`, `queries = [[0,1,2],[1,2,1],[0,2,4]]` → `[6,2,12]`.

**Ví dụ 3:** `nums = [3,6]`, `queries = [[0,1,1],[1,1,3]]` → `[2,8]`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^9` (tăng ngặt), `1 ≤ q ≤ 10^5`, `0 ≤ l ≤ r < n`, `1 ≤ k ≤ 10^9`.

In [1]:
from bisect import bisect_left, bisect_right
import random, time

examples = [
    ([1, 4, 7], [[0, 2, 1], [1, 1, 2], [0, 0, 3]], [2, 6, 6]),
    ([2, 5, 8], [[0, 1, 2], [1, 2, 1], [0, 2, 4]], [6, 2, 12]),
    ([3, 6], [[0, 1, 1], [1, 1, 3]], [2, 8]),
]
for nums, qs, exp in examples:
    print(nums, qs, "-> kỳ vọng", exp)

[1, 4, 7] [[0, 2, 1], [1, 1, 2], [0, 0, 3]] -> kỳ vọng [2, 6, 6]
[2, 5, 8] [[0, 1, 2], [1, 2, 1], [0, 2, 4]] -> kỳ vọng [6, 2, 12]
[3, 6] [[0, 1, 1], [1, 1, 3]] -> kỳ vọng [2, 8]


## 3. Kiến thức nền

### 3.1. `bisect_left`, `bisect_right` và tham số `lo`, `hi`

- `bisect_left(a, x)`: số phần tử `< x` (vị trí chèn bên trái).
- `bisect_right(a, x)`: số phần tử `≤ x`.
- Cả hai nhận thêm `lo, hi` để chỉ tìm trong **đoạn con** `a[lo:hi]` mà **không cần cắt mảng** (cắt mảng tốn `O(len)`). Kết quả vẫn là chỉ số trong mảng gốc.

In [2]:
a = [1, 3, 3, 5, 8, 9]
print(bisect_left(a, 3), bisect_right(a, 3))
print("trong đoạn [2, 5):", bisect_right(a, 5, 2, 5), "(chỉ số trong mảng gốc)")

1 3
trong đoạn [2, 5): 4 (chỉ số trong mảng gốc)


### 3.2. Bài toán "số nguyên dương thứ k bị thiếu"

Cho `S` tăng ngặt các số nguyên dương. Số bị thiếu trước (và không kể) `S[i]` là `S[i] - (i + 1)` (vì từ `1..S[i]` có `S[i]` số, trong đó `i + 1` số có trong `S`).

- Dãy `S[i] - i` **không giảm** (vì `S[i+1] ≥ S[i] + 1`).
- Gọi `c` = số chỉ số `i` mà `S[i] - i ≤ k` (tương đương `S[i] - (i+1) < k`). Khi đó **đáp án = k + c**.

Trực giác: đi lần lượt `1, 2, 3, …`; mỗi khi gặp một phần tử của `S` "đứng chắn" trước đáp án thì đáp án bị đẩy lên 1.

In [3]:
def kth_missing(S, k):
    lo, hi = 0, len(S)
    while lo < hi:
        mid = (lo + hi) // 2
        if S[mid] - mid <= k:
            lo = mid + 1
        else:
            hi = mid
    return k + lo

S = [2, 3, 4, 7, 11]
print([kth_missing(S, k) for k in range(1, 8)])     # các số thiếu: 1,5,6,8,9,10,12

[1, 5, 6, 8, 9, 10, 12]


### 3.3. Chia đôi số chẵn

Số chẵn thứ `t` là `2t`. Xóa một tập số chẵn `E` khỏi dãy chẵn ⇔ xóa tập `H = {e / 2 : e ∈ E}` khỏi dãy `1, 2, 3, …`. Vậy số chẵn còn lại thứ `k` = `2 × (số nguyên dương thứ k không thuộc H)`.

In [4]:
E = [4, 8]
H = [e // 2 for e in E]
print("H =", H, "| chẵn còn lại thứ 1..5:", [2 * kth_missing(H, k) for k in range(1, 6)])

H = [2, 4] | chẵn còn lại thứ 1..5: [2, 6, 10, 12, 14]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay truy vấn thứ 3 của ví dụ 2: `nums = [2,5,8]`, `[0, 2, 4]`.

- Đoạn con `[2, 5, 8]`; số chẵn bị xóa: `2, 8`.
- Dãy chẵn còn lại: `4, 6, 10, 12, 14, …` → thứ 4 là `12`.
- Cách chia đôi: `H = [1, 4]`. Số nguyên dương không thuộc `H`: `2, 3, 5, 6, …` → thứ 4 là `6` → nhân 2 = `12` ✔.
- Công thức: `S[i] - i` = `[1 - 0, 4 - 1] = [1, 3]`, đều `≤ 4` ⇒ `c = 2` ⇒ `k + c = 6` ⇒ `12` ✔.

**Điểm mấu chốt:** vì `nums` tăng ngặt, các số chẵn trong đoạn `nums[l..r]` là một **đoạn liên tiếp** trong danh sách các số chẵn của toàn mảng (giữ nguyên thứ tự). Ta chỉ cần biết đoạn đó là `[a, b)` trong danh sách chẵn.

## 5. Cách 1 — Brute Force: mô phỏng trực tiếp

Với mỗi truy vấn, đưa các phần tử của đoạn vào `set`, rồi đếm dần `2, 4, 6, …` bỏ qua phần tử trong set, tới khi được `k` số. Thời gian mỗi truy vấn `O((r - l) + k)` — `k` tới `10^9` là bất khả.

In [5]:
def brute(nums, queries):
    ans = []
    for l, r, k in queries:
        removed = set(nums[l:r + 1])
        x, cnt = 0, 0
        while cnt < k:
            x += 2
            if x not in removed:
                cnt += 1
        ans.append(x)
    return ans

for nums, qs, exp in examples:
    print(brute(nums, qs), "kỳ vọng", exp)

[2, 6, 6] kỳ vọng [2, 6, 6]
[6, 2, 12] kỳ vọng [6, 2, 12]
[2, 8] kỳ vọng [2, 8]


**Nút thắt:** (1) phụ thuộc `k`; (2) mỗi truy vấn duyệt lại cả đoạn con. Cần bỏ phụ thuộc vào `k` (dùng công thức `k + c`) và bỏ phụ thuộc vào độ dài đoạn (tiền xử lý toàn cục).

## 6. Ý tưởng thuật toán / bước đột phá

**Tiền xử lý:** duyệt `nums`, lấy các số chẵn theo thứ tự: vị trí `P[j]` (chỉ số trong `nums`) và nửa giá trị `H[j] = nums[P[j]] / 2`. `H` tăng ngặt.

**Với truy vấn `[l, r, k]`:**

1. `a = bisect_left(P, l)`, `b = bisect_right(P, r)` ⇒ các số chẵn trong đoạn là `H[a:b]`.
2. Đặt `S[i] = H[a + i]` (chỉ số cục bộ `i`). Cần `c = #{i : S[i] - i ≤ k}`.
3. **Đột phá:** `S[i] - i = H[a+i] - (a+i) + a`. Định nghĩa **toàn cục** `G[j] = H[j] - j` (không giảm). Khi đó điều kiện ⇔ `G[j] ≤ k - a` với `j ∈ [a, b)`.
4. `c = bisect_right(G, k - a, a, b) - a`. Đáp án `= 2 · (k + c)`.

Mỗi truy vấn chỉ 3 lần `bisect` ⇒ `O(log n)`.

**Quy tắc "nếu … thì …":** nếu đoạn không có số chẵn (`a == b`) thì `c = 0`, đáp án `2k`.

## 7. Các cách tiếp theo

- **Cách 2:** mỗi truy vấn lọc số chẵn trong đoạn rồi quét tuyến tính `S[i] - i ≤ k` — `O(r - l)` mỗi truy vấn.
- **Cách 3:** lọc như Cách 2 nhưng dùng `kth_missing` (nhị phân) — vẫn `O(r - l)` do bước lọc.
- **Cách 4:** **tìm nhị phân trên đáp án**: đếm số chẵn bị xóa `≤ x` bằng `bisect` trên `H[a:b]` — `O(log k · log n)` mỗi truy vấn.
- **Cách 5:** mảng toàn cục `G` + `bisect` có `lo/hi` — `O(log n)` mỗi truy vấn (lời giải chính).

### 7.1. Cách 2 — Lọc đoạn + quét tuyến tính

In [6]:
def per_query_linear(nums, queries):
    ans = []
    for l, r, k in queries:
        S = [x // 2 for x in nums[l:r + 1] if x % 2 == 0]
        c = 0
        for i, s in enumerate(S):
            if s - i <= k:
                c += 1
            else:
                break                   # S[i]-i không giảm nên dừng được
        ans.append(2 * (k + c))
    return ans

for nums, qs, exp in examples:
    print(per_query_linear(nums, qs), "kỳ vọng", exp)

[2, 6, 6] kỳ vọng [2, 6, 6]
[6, 2, 12] kỳ vọng [6, 2, 12]
[2, 8] kỳ vọng [2, 8]


**Độ phức tạp:** `O(Σ (r - l + 1))` — tệ nhất `O(n · q) = 10^{10}`.

### 7.2. Cách 3 — Lọc đoạn + nhị phân "số thứ k bị thiếu"

In [7]:
def per_query_binary(nums, queries):
    ans = []
    for l, r, k in queries:
        S = [x // 2 for x in nums[l:r + 1] if x % 2 == 0]
        ans.append(2 * kth_missing(S, k))
    return ans

for nums, qs, exp in examples:
    print(per_query_binary(nums, qs), "kỳ vọng", exp)

[2, 6, 6] kỳ vọng [2, 6, 6]
[6, 2, 12] kỳ vọng [6, 2, 12]
[2, 8] kỳ vọng [2, 8]


Phần nhị phân chỉ `O(log n)`, nhưng bước **lọc** vẫn `O(r - l)`. Muốn nhanh thật sự phải bỏ bước lọc ⇒ tiền xử lý toàn cục.

### 7.3. Cách 4 — Tìm nhị phân trên đáp án

Gọi đáp án là `2m`. Số chẵn còn lại `≤ 2m` là `m - cnt(m)` với `cnt(m)` = số phần tử `H[a:b]` mà `≤ m` = `bisect_right(H, m, a, b) - a`. Hàm `m - cnt(m)` không giảm theo `m`. Ta tìm `m` nhỏ nhất với `m - cnt(m) ≥ k`. Biết `k ≤ m ≤ k + (b - a)`.

In [8]:
def preprocess(nums):
    P, H = [], []
    for i, x in enumerate(nums):
        if x % 2 == 0:
            P.append(i)
            H.append(x // 2)
    return P, H

def answer_binary_search(nums, queries):
    P, H = preprocess(nums)
    ans = []
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        lo, hi = k, k + (b - a)
        while lo < hi:
            mid = (lo + hi) // 2
            if mid - (bisect_right(H, mid, a, b) - a) >= k:
                hi = mid
            else:
                lo = mid + 1
        ans.append(2 * lo)
    return ans

for nums, qs, exp in examples:
    print(answer_binary_search(nums, qs), "kỳ vọng", exp)

[2, 6, 6] kỳ vọng [2, 6, 6]
[6, 2, 12] kỳ vọng [6, 2, 12]
[2, 8] kỳ vọng [2, 8]


In [9]:
class SolutionAnswerBS:
    def kthRemainingInteger(self, nums: list[int], queries: list[list[int]]) -> list[int]:
        return answer_binary_search(nums, queries)

print([SolutionAnswerBS().kthRemainingInteger(a, q) for a, q, _ in examples])

[[2, 6, 6], [6, 2, 12], [2, 8]]


**Độ phức tạp Cách 4:** `O(n + q · log n · log n)` (mỗi bước nhị phân ngoài gọi `bisect` trong). Đủ nhanh, nhưng Cách 5 gọn hơn.

### 7.4. Cách 5 — Mảng toàn cục `G[j] = H[j] - j` (lời giải chính)

**Giải thích từng đoạn:**

1. Tiền xử lý `P`, `H`, rồi `G[j] = H[j] - j`.
2. Mỗi truy vấn: `a, b` như trên; `c = bisect_right(G, k - a, a, b) - a`; đáp án `2 (k + c)`.

Vì sao `G` không giảm: `H[j+1] ≥ H[j] + 1` ⇒ `G[j+1] = H[j+1] - j - 1 ≥ H[j] - j = G[j]`.

In [10]:
def global_G(nums, queries):
    P, H = preprocess(nums)
    G = [h - j for j, h in enumerate(H)]
    ans = []
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        c = bisect_right(G, k - a, a, b) - a
        ans.append(2 * (k + c))
    return ans

for nums, qs, exp in examples:
    print(global_G(nums, qs), "kỳ vọng", exp)

[2, 6, 6] kỳ vọng [2, 6, 6]
[6, 2, 12] kỳ vọng [6, 2, 12]
[2, 8] kỳ vọng [2, 8]


**Độ phức tạp Cách 5:** `O(n + q log n)` thời gian, `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 5 trên `nums = [2, 3, 4, 6, 9, 10, 14]` (ví dụ tự dựng) và vài truy vấn.

Tiền xử lý:

| j | P[j] (chỉ số) | số chẵn | H[j] | G[j] = H[j] − j |
|---|---|---|---|---|
| 0 | 0 | 2 | 1 | 1 |
| 1 | 2 | 4 | 2 | 1 |
| 2 | 3 | 6 | 3 | 1 |
| 3 | 5 | 10 | 5 | 2 |
| 4 | 6 | 14 | 7 | 3 |

Truy vấn `[2, 6, 3]`: `a = 1`, `b = 5`. Cần `G[j] ≤ k - a = 2` với `j ∈ [1,5)`: `G = 1, 1, 2, 3` ⇒ 3 chỉ số ⇒ `c = 3` ⇒ đáp án `2·(3+3) = 12`. Kiểm tra: đoạn `[4,6,9,10,14]` xóa `4,6,10,14`; dãy còn `2, 8, 12, 16, …` → thứ 3 là `12` ✔.

In [11]:
def trace(nums, queries):
    P, H = preprocess(nums)
    G = [h - j for j, h in enumerate(H)]
    print("P =", P); print("H =", H); print("G =", G)
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        c = bisect_right(G, k - a, a, b) - a
        print(f"truy vấn l={l} r={r} k={k}: a={a} b={b} G[a:b]={G[a:b]} ngưỡng k-a={k-a} -> c={c} -> đáp án {2*(k+c)}")

nums = [2, 3, 4, 6, 9, 10, 14]
qs = [[2, 6, 3], [0, 6, 1], [1, 1, 5], [0, 3, 2]]
trace(nums, qs)
print("brute:", brute(nums, qs))

P = [0, 2, 3, 5, 6]
H = [1, 2, 3, 5, 7]
G = [1, 1, 1, 2, 3]
truy vấn l=2 r=6 k=3: a=1 b=5 G[a:b]=[1, 1, 2, 3] ngưỡng k-a=2 -> c=3 -> đáp án 12
truy vấn l=0 r=6 k=1: a=0 b=5 G[a:b]=[1, 1, 1, 2, 3] ngưỡng k-a=1 -> c=3 -> đáp án 8
truy vấn l=1 r=1 k=5: a=1 b=1 G[a:b]=[] ngưỡng k-a=4 -> c=0 -> đáp án 10
truy vấn l=0 r=3 k=2: a=0 b=3 G[a:b]=[1, 1, 1] ngưỡng k-a=2 -> c=3 -> đáp án 10
brute: [12, 8, 10, 10]


In [12]:
# Trace "đi bộ" cho truy vấn [2, 6, 3]: liệt kê số chẵn và đánh dấu bị xóa
nums = [2, 3, 4, 6, 9, 10, 14]
removed = set(nums[2:7])
cnt = 0
for x in range(2, 20, 2):
    if x in removed:
        print(x, "bị xóa")
    else:
        cnt += 1
        print(x, f"còn lại (thứ {cnt})")

2 còn lại (thứ 1)
4 bị xóa
6 bị xóa
8 còn lại (thứ 2)
10 bị xóa
12 còn lại (thứ 3)
14 bị xóa
16 còn lại (thứ 4)
18 còn lại (thứ 5)


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def kthRemainingInteger(self, nums: list[int], queries: list[list[int]]) -> list[int]:
        P = []          # chỉ số các phần tử chẵn trong nums
        G = []          # G[j] = nums[P[j]]//2 - j  (không giảm)
        for i, x in enumerate(nums):
            if x % 2 == 0:
                G.append(x // 2 - len(P))
                P.append(i)
        ans = []
        for l, r, k in queries:
            a = bisect_left(P, l)
            b = bisect_right(P, r)
            c = bisect_right(G, k - a, a, b) - a
            ans.append(2 * (k + c))
        return ans

sol = Solution()
for nums, qs, exp in examples:
    got = sol.kthRemainingInteger(nums, qs)
    print(got, "✔" if got == exp else "✘")

[2, 6, 6] ✔
[6, 2, 12] ✔
[2, 8] ✔


## 10. Vì sao thuật toán đúng

1. **Quy về số thiếu:** ánh xạ `e ↦ e/2` là song ánh giữa số chẵn dương và số nguyên dương, bảo toàn thứ tự. Nên số chẵn còn lại thứ `k` = `2 ×` (số nguyên dương thứ `k` không thuộc `S`), với `S` = nửa các số chẵn trong đoạn.
2. **`S` là đoạn liên tiếp của `H`:** `nums` tăng ngặt nên các phần tử chẵn của `nums[l..r]` chính là các `P[j] ∈ [l, r]`, tức `j ∈ [a, b)`, theo đúng thứ tự tăng.
3. **Công thức `k + c`:** gọi `x` là đáp án (số nguyên dương thứ `k` không thuộc `S`). Có đúng `k - 1` số bị thiếu `< x`, nên số phần tử của `S` nhỏ hơn `x` là `x - 1 - (k - 1) = x - k`. Phần tử `S[i] < x` ⇔ số thiếu trước `S[i]` (`= S[i] - i - 1`) `< k` ⇔ `S[i] - i ≤ k`. Vậy `c = x - k`.
4. **Đơn điệu:** `S[i] - i` không giảm nên tập `{i : S[i] - i ≤ k}` là một tiền tố, đếm bằng `bisect_right`. Dịch chỉ số `S[i] - i = G[a + i] + a` cho điều kiện `G[j] ≤ k - a`.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `q = len(queries)`, `L = r - l + 1`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(Σ (L + k))` | `O(L)` |
| 2. Lọc + quét | `O(Σ L)` ≤ `O(n q)` | `O(n)` |
| 3. Lọc + nhị phân | `O(Σ L)` | `O(n)` |
| 4. Nhị phân đáp án | `O(n + q log² n)` | `O(n)` |
| 5. Mảng `G` toàn cục | `O(n + q log n)` | `O(n)` |

## 12. Bộ test

In [14]:
def run_all(nums, qs, with_brute=True):
    res = [per_query_linear(nums, qs), per_query_binary(nums, qs), answer_binary_search(nums, qs),
           global_G(nums, qs), Solution().kthRemainingInteger(nums, qs)]
    if with_brute:
        res.append(brute(nums, qs))
    return res

for nums, qs, exp in examples:
    assert all(r == exp for r in run_all(nums, qs)), run_all(nums, qs)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [15]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng mô phỏng)
cases = [
    ([1], [[0, 0, 1]]),                    # không có số chẵn -> 2k
    ([2], [[0, 0, 1], [0, 0, 2]]),          # xóa 2 -> 4, 6
    ([2, 4, 6, 8], [[0, 3, 1], [0, 3, 5]]), # xóa cả dãy đầu -> 10, 18
    ([1, 3, 5], [[0, 2, 10**9]]),          # k cực lớn, không số chẵn
    ([10**9], [[0, 0, 5 * 10**8]]),        # 10^9 là số chẵn thứ 5*10^8 -> bị đẩy lên
]
for nums, qs in cases:
    fast = Solution().kthRemainingInteger(nums, qs)
    small = [q for q in qs if q[2] <= 10**4]
    if small:
        assert Solution().kthRemainingInteger(nums, small) == brute(nums, small)
    print(nums, qs, "->", fast)
assert Solution().kthRemainingInteger([10**9], [[0, 0, 5 * 10**8]]) == [10**9 + 2]
assert Solution().kthRemainingInteger([1, 3, 5], [[0, 2, 10**9]]) == [2 * 10**9]
print("Ca biên: OK")

[1] [[0, 0, 1]] -> [2]
[2] [[0, 0, 1], [0, 0, 2]] -> [4, 6]
[2, 4, 6, 8] [[0, 3, 1], [0, 3, 5]] -> [10, 18]
[1, 3, 5] [[0, 2, 1000000000]] -> [2000000000]
[1000000000] [[0, 0, 500000000]] -> [1000000002]
Ca biên: OK


In [16]:
# 12.3 phản ví dụ cho ý tưởng sai "đáp án = 2k + 2 * (số chẵn bị xóa)"
nums, qs = [2, 100], [[0, 1, 1]]
wrong = [2 * k + 2 * sum(1 for x in nums[l:r + 1] if x % 2 == 0) for l, r, k in qs]
print("sai:", wrong, "| đúng:", Solution().kthRemainingInteger(nums, qs))

sai: [6] | đúng: [4]


In [17]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3911)
for _ in range(300):
    n = rng.randint(1, 12)
    nums = sorted(rng.sample(range(1, 40), n))
    qs = []
    for _ in range(rng.randint(1, 8)):
        l = rng.randrange(n); r = rng.randrange(l, n)
        qs.append([l, r, rng.randint(1, 30)])
    res = run_all(nums, qs)
    assert all(x == res[0] for x in res), (nums, qs, res)
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [18]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(11)
n = 10**5
nums = sorted(rng.sample(range(1, 10**9), n))
qs = []
for _ in range(10**5):
    l = rng.randrange(n); r = rng.randrange(l, n)
    qs.append([l, r, rng.randint(1, 10**9)])
t0 = time.time(); A = Solution().kthRemainingInteger(nums, qs); t1 = time.time()
B = answer_binary_search(nums, qs[:2000])
print("khớp Cách 4 trên 2000 truy vấn đầu:", A[:2000] == B, "| thời gian < 2s:", t1 - t0 < 2)

khớp Cách 4 trên 2000 truy vấn đầu: True | thời gian < 2s: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng toàn bộ số chẵn bị xóa thay vì chỉ những số "chắn trước" đáp án

Chỉ số chẵn bị xóa **nhỏ hơn đáp án** mới đẩy đáp án lên. Số bị xóa lớn hơn không ảnh hưởng.

In [19]:
def wrong_all_removed(nums, queries):
    return [2 * (k + sum(1 for x in nums[l:r + 1] if x % 2 == 0)) for l, r, k in queries]

nums, qs = [4, 100], [[0, 1, 1]]
print("sai:", wrong_all_removed(nums, qs), "| đúng:", Solution().kthRemainingInteger(nums, qs))

sai: [6] | đúng: [2]


**Vì sao sai:** số `100` không nằm trước đáp án `2` nên không được đếm. **Sửa:** chỉ đếm `S[i] - i ≤ k`.

### Lỗi 2 — Dùng `bisect_right(G, k, a, b)` mà quên dịch `- a`

`G[j] = H[j] - j` dùng chỉ số **toàn cục**. Trong đoạn con, chỉ số cục bộ là `j - a`, nên điều kiện là `G[j] ≤ k - a`, không phải `G[j] ≤ k`.

In [20]:
def wrong_no_shift(nums, queries):
    P, H = preprocess(nums)
    G = [h - j for j, h in enumerate(H)]
    ans = []
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        c = bisect_right(G, k, a, b) - a          # SAI: thiếu "- a" ở ngưỡng
        ans.append(2 * (k + c))
    return ans

nums, qs = [2, 4, 6, 8], [[2, 3, 1]]
print("sai:", wrong_no_shift(nums, qs), "| đúng:", Solution().kthRemainingInteger(nums, qs), "| brute:", brute(nums, qs))

sai: [6] | đúng: [2] | brute: [2]


### Lỗi 3 — Cắt mảng `G[a:b]` trong mỗi truy vấn

`bisect_right(G[a:b], …)` cho kết quả đúng nhưng **cắt mảng tốn `O(b - a)`**, làm mất lợi thế `O(log n)`. Hãy dùng tham số `lo, hi`.

In [21]:
def slow_slice(nums, queries):
    P, H = preprocess(nums)
    G = [h - j for j, h in enumerate(H)]
    out = []
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        out.append(2 * (k + bisect_right(G[a:b], k - a)))   # đúng nhưng O(b-a)
    return out

rng = random.Random(2)
n = 10**5
nums = list(range(2, 2 * n + 2, 2))           # toàn số chẵn
qs = [[0, n - 1, rng.randint(1, 10**9)] for _ in range(300)]
t0 = time.time(); x1 = slow_slice(nums, qs); t1 = time.time(); x2 = Solution().kthRemainingInteger(nums, qs); t2 = time.time()
print("kết quả giống nhau:", x1 == x2, "| cắt mảng chậm hơn:", (t1 - t0) > (t2 - t1))

kết quả giống nhau: True | cắt mảng chậm hơn: True


### Lỗi 4 — Quên rằng số lẻ trong `nums` không xóa gì

Đưa cả số lẻ vào `S` sẽ làm đếm sai (số lẻ không có trong dãy chẵn).

In [22]:
def wrong_include_odd(nums, queries):
    ans = []
    for l, r, k in queries:
        S = [(x + 1) // 2 for x in nums[l:r + 1]]   # SAI: số lẻ cũng bị coi là "xóa"
        ans.append(2 * kth_missing(S, k))
    return ans

nums, qs = [1, 3, 5], [[0, 2, 1]]
print("sai:", wrong_include_odd(nums, qs), "| đúng:", Solution().kthRemainingInteger(nums, qs))

sai: [8] | đúng: [2]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Phần tử thứ k còn lại sau khi xóa một tập đã sắp xếp"** ⇒ công thức `k + #{i : S[i] - i ≤ k}` (LC 1539).
- **Dãy số học (chẵn, bội của d, …)** ⇒ chia cho bước để đưa về `1, 2, 3, …`.
- **Truy vấn đoạn trên mảng tăng ngặt** ⇒ phần tử thỏa tính chất trong đoạn là một đoạn liên tiếp của danh sách lọc toàn cục ⇒ `bisect` hai đầu.
- **Đại lượng phụ thuộc chỉ số cục bộ** (`S[i] - i`) ⇒ viết lại theo chỉ số toàn cục + hằng số dịch (`G[j] + a`) để tiền xử lý một lần.

## 15. Biến thể của bài

1. **Dãy bội của `d`** (`d, 2d, 3d, …`): thay `2` bằng `d`, lọc `x % d == 0`, `H = x // d`.
2. **Dãy mọi số nguyên dương** (không chỉ chẵn): `d = 1` — chính là LC 1539 trên đoạn.
3. **`nums` không sắp xếp**: cần cấu trúc mạnh hơn (merge sort tree hoặc persistent segment tree) để đếm số phần tử `≤ x` trong đoạn chỉ số.
4. **Có cập nhật** (thêm/xóa phần tử): Fenwick tree trên giá trị nén + tìm nhị phân trên cây.

In [23]:
# Biến thể 1: bội của d
def kth_remaining_multiple(nums, queries, d):
    P, G = [], []
    for i, x in enumerate(nums):
        if x % d == 0:
            G.append(x // d - len(P)); P.append(i)
    out = []
    for l, r, k in queries:
        a, b = bisect_left(P, l), bisect_right(P, r)
        out.append(d * (k + bisect_right(G, k - a, a, b) - a))
    return out

def brute_multiple(nums, queries, d):
    out = []
    for l, r, k in queries:
        rem = set(nums[l:r + 1]); x = 0; c = 0
        while c < k:
            x += d
            if x not in rem: c += 1
        out.append(x)
    return out

rng = random.Random(15)
for _ in range(200):
    n = rng.randint(1, 10); nums = sorted(rng.sample(range(1, 60), n)); d = rng.randint(1, 5)
    qs = [[l, rng.randrange(l, n), rng.randint(1, 20)] for l in [rng.randrange(n) for _ in range(5)]]
    assert kth_remaining_multiple(nums, qs, d) == brute_multiple(nums, qs, d)
print("Biến thể 1 khớp mô phỏng.")

Biến thể 1 khớp mô phỏng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | Đếm từng số chẵn | `O(L + k)` | `O(L)` | Kiểm thử |
| 2. Lọc + quét | `k + c`, quét tuyến tính | `O(L)` | `O(L)` | Đoạn ngắn |
| 3. Lọc + nhị phân | LC 1539 trên đoạn | `O(L)` | `O(L)` | Đoạn ngắn |
| 4. Nhị phân đáp án | `m - cnt(m) ≥ k` | `O(log² n)` | `O(n)` | Khi `cnt` dễ tính, khó có công thức |
| 5. `G` toàn cục | `bisect_right(G, k-a, a, b)` | `O(log n)` | `O(n)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nums = [2, 4, 6]`, truy vấn `[0, 2, 2]` cho đáp án bao nhiêu?

**Bài 2 (tính tay).** Với `nums = [1, 6, 8, 12]`, truy vấn `[1, 3, 4]`: lập `S`, `S[i] - i`, tính `c` và đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `kth_missing_positive(arr, k)` (LC 1539) bằng nhị phân.

**Bài 4 (tự viết lại).** Viết lại lời giải chính không dùng `bisect` (tự cài vòng nhị phân cho cả ba lần tìm).

**Bài 5 (mini challenge).** Mỗi truy vấn có thêm `d` (dãy bội của `d` thay vì số chẵn), với `d ∈ {1,…,5}`. Tiền xử lý cho mọi `d` và trả lời truy vấn trong `O(log n)`.

In [24]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def kth_missing_positive(arr, k):
    # TODO: lo, hi = 0, len(arr); tìm số chỉ số i với arr[i] - i <= k
    pass

In [25]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def multi_d(nums, queries):
    # TODO: với mỗi d trong 1..5 dựng (P_d, G_d); truy vấn [l, r, k, d]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Xóa `2, 4, 6`. (2) Còn `8, 10, …`. (3) Thứ 2 là `10`.

**Bài 2.** (1) Đoạn `[6, 8, 12]` ⇒ `S = [3, 4, 6]`. (2) `S[i] - i = [3, 3, 4]`, đều `≤ 4` ⇒ `c = 3`. (3) Đáp án `2·(4+3) = 14`.

**Bài 3.** (1) Điều kiện `arr[mid] - mid <= k` thì sang phải. (2) Trả về `k + lo`. (3) Thử với `[2,3,4,7,11], k = 5` → `9`.

**Bài 4.** (1) Viết hàm `lower(a, x, lo, hi)` và `upper(a, x, lo, hi)`. (2) Thay ba lời gọi `bisect`. (3) Đối chiếu ngẫu nhiên.

**Bài 5.** (1) Lặp `d` từ 1 tới 5 khi tiền xử lý. (2) Lưu trong dict theo `d`. (3) Đáp án `d·(k + c)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
assert Solution().kthRemainingInteger([2, 4, 6], [[0, 2, 2]]) == [10]
assert Solution().kthRemainingInteger([1, 6, 8, 12], [[1, 3, 4]]) == [14] == brute([1, 6, 8, 12], [[1, 3, 4]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [27]:
# Bài 3
def kth_missing_positive_sol(arr, k):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - mid <= k:
            lo = mid + 1
        else:
            hi = mid
    return k + lo

assert kth_missing_positive_sol([2, 3, 4, 7, 11], 5) == 9
assert kth_missing_positive_sol([1, 2, 3, 4], 2) == 6
print("Bài 3 OK")

Bài 3 OK


In [28]:
# Bài 4
def lower(a, x, lo, hi):
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x: lo = mid + 1
        else: hi = mid
    return lo

def upper(a, x, lo, hi):
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] <= x: lo = mid + 1
        else: hi = mid
    return lo

def sol_no_bisect(nums, queries):
    P, G = [], []
    for i, x in enumerate(nums):
        if x % 2 == 0:
            G.append(x // 2 - len(P)); P.append(i)
    out = []
    for l, r, k in queries:
        a, b = lower(P, l, 0, len(P)), upper(P, r, 0, len(P))
        out.append(2 * (k + upper(G, k - a, a, b) - a))
    return out

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(1, 15); nums = sorted(rng.sample(range(1, 60), n))
    qs = [[l, rng.randrange(l, n), rng.randint(1, 40)] for l in [rng.randrange(n) for _ in range(6)]]
    assert sol_no_bisect(nums, qs) == Solution().kthRemainingInteger(nums, qs)
print("Bài 4 OK")

Bài 4 OK


In [29]:
# Bài 5
def multi_d_sol(nums, queries):
    pre = {}
    for d in range(1, 6):
        P, G = [], []
        for i, x in enumerate(nums):
            if x % d == 0:
                G.append(x // d - len(P)); P.append(i)
        pre[d] = (P, G)
    out = []
    for l, r, k, d in queries:
        P, G = pre[d]
        a, b = bisect_left(P, l), bisect_right(P, r)
        out.append(d * (k + bisect_right(G, k - a, a, b) - a))
    return out

rng = random.Random(5)
for _ in range(200):
    n = rng.randint(1, 10); nums = sorted(rng.sample(range(1, 50), n))
    qs = []
    for _ in range(5):
        l = rng.randrange(n); qs.append([l, rng.randrange(l, n), rng.randint(1, 15), rng.randint(1, 5)])
    assert multi_d_sol(nums, qs) == [brute_multiple(nums, [[l, r, k]], d)[0] for l, r, k, d in qs]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
P, G = [], []                       # vị trí số chẵn, G[j] = x//2 - j
for i, x in enumerate(nums):
    if x % 2 == 0:
        G.append(x // 2 - len(P)); P.append(i)
for l, r, k in queries:
    a = bisect_left(P, l); b = bisect_right(P, r)
    c = bisect_right(G, k - a, a, b) - a
    ans.append(2 * (k + c))
```

**Checklist:**

- [ ] Chỉ lấy **số chẵn** của đoạn?
- [ ] Chia đôi để quy về số nguyên dương thứ `k` bị thiếu?
- [ ] Ngưỡng là `k - a` (dịch chỉ số toàn cục → cục bộ)?
- [ ] Dùng `bisect` với `lo, hi`, không cắt mảng?
- [ ] Nhân lại `2` ở cuối?